# Модуль 10.7 — Деплой агента: HF Spaces

Домашка к [лекции 10.7](https://itrubnikov.github.io/Train_of_Thought/docs/modules/10-7-deploy-agent/). Соберёте smolagents-агента с инструментом, прогоните его, добавите **свой** инструмент — и развернёте как живой веб-чат на HF Spaces по трём файлам. Нужен бесплатный HF-токен (см. README).

## 0. Установка и токен

Запустите ячейку. В Colab токен берётся из Secrets (значок ключа слева, имя
`HF_TOKEN`); локально — из `.env`. smolagents-модель сама возьмёт ключ из
окружения `HF_TOKEN`.

In [ ]:
!pip -q install smolagents python-dotenv
import os

# токен: Colab Secrets -> переменная окружения; иначе .env
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

if not os.getenv("HF_TOKEN"):
    raise RuntimeError("Нет HF_TOKEN. Заведите бесплатный токен на huggingface.co "
                       "(Settings -> Access Tokens) и впишите в .env или Colab Secrets.")
print("Готово, токен на месте.")

## 1. Первый агент с инструментом

Инструмент — обычная Python-функция с декоратором `@tool` и **докстрингом**:
smolagents строит из докстринга описание инструмента для модели (без докстринга
агент не поймёт, когда его звать). `CodeAgent` пишет действия кодом — он сам
сгенерирует вызов `word_count(...)`, выполнит и вернёт ответ. Модель — открытая,
через HF Inference.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool

@tool
def word_count(text: str) -> int:
    """Считает количество слов в тексте.

    Args:
        text: текст, в котором нужно посчитать слова.
    """
    return len(text.split())

model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")  # открытая модель
agent = CodeAgent(tools=[word_count], model=model)

print(agent.run("Сколько слов во фразе 'агент это петля из шагов'? Посчитай инструментом."))

## 2. Добавляем свой инструмент

Вот второй инструмент — конвертер температуры (детерминированный, без интернета).
Даём агенту оба инструмента и видим в логах, что на нужный вопрос он выбирает
именно `c_to_f`. Это и есть «свой `@tool`» из домашки — замените на любую
полезную функцию.

In [ ]:
@tool
def c_to_f(celsius: float) -> float:
    """Переводит температуру из Цельсия в Фаренгейты.

    Args:
        celsius: температура в градусах Цельсия.
    """
    return celsius * 9 / 5 + 32

agent2 = CodeAgent(tools=[word_count, c_to_f], model=model)
print(agent2.run("Сколько будет 25 градусов Цельсия в Фаренгейтах? Используй инструмент."))

## 3. Из агента — в приложение (три файла Space)

Чтобы агент стал веб-чатом, к нему добавляется ровно одна строка —
`GradioUI(agent).launch()`. Это и есть `app.py`. Ниже — содержимое трёх файлов
Space. (В ноутбуке мы `.launch()` не зовём — он поднимает сервер; запустится он
уже на Spaces.)

**`app.py`:**
```python
from smolagents import CodeAgent, InferenceClientModel, tool, GradioUI

@tool
def word_count(text: str) -> int:
    """Считает количество слов в тексте.

    Args:
        text: текст, в котором нужно посчитать слова.
    """
    return len(text.split())

model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")
agent = CodeAgent(tools=[word_count], model=model)
GradioUI(agent).launch()
```

**`requirements.txt`:**
```text
smolagents
gradio>=5,<6
```

**`README.md`** (YAML-шапка = метаданные сборки):
```yaml
---
title: My First Agent
emoji: 🤖
colorFrom: blue
colorTo: green
sdk: gradio
sdk_version: 5.49.1
app_file: app.py
---
```

Готовые файлы лежат в репозитории: `spaces/module-10-7-agent/`.

## 4. Деплой за три клика

1. На huggingface.co: **New → Space**, SDK — **Gradio**.
2. Залейте `app.py` / `requirements.txt` / `README.md` (через *Files → Add file*
   или `git push` в репозиторий Space). Файлы — из `spaces/module-10-7-agent/`,
   подставив **свой** инструмент.
3. **Settings → Variables and secrets → New secret**: `HF_TOKEN` = ваш бесплатный
   токен.
4. Space соберётся сам → откройте URL → задайте агенту вопрос в чате.

Грабли (из лекции): `sdk_version` обязательно **5.x** (gradio 4.x падает на
Python 3.13 из-за `audioop`); без секрета `HF_TOKEN` агент молчит; бесплатный
Space засыпает — первый ответ после простоя идёт дольше.

## Задачи — доработайте рабочий код

Сделайте минимум 3 из 4:

1. **Свой инструмент.** Напишите `@tool` с докстрингом (конвертер единиц,
   «длина строки», «реверс текста» — что угодно полезное) и убедитесь, что агент
   его вызывает на подходящем вопросе.
2. **Два инструмента.** Дайте агенту два инструмента и задайте вопрос, требующий
   обоих по очереди; в логах посмотрите, как он их комбинирует.
3. **Деплой.** Создайте Space из трёх файлов с вашим инструментом, добавьте
   секрет `HF_TOKEN`, дождитесь сборки и откройте чат по URL.
4. **Сломайте и почините.** Поставьте в `README.md` `sdk_version: 4.44.0`,
   задеплойте, посмотрите краш `audioop` в логах, верните `5.x` — почувствуйте
   грабли версий на себе.

По каждой задаче запишите короткий вывод; для задачи 3 — ссылку на ваш Space.

## Что сдать

- [ ] Ноутбук прогнан (`Run all`) — агент с инструментом ответил.
- [ ] Добавлен **свой** `@tool`; в логах видно, что агент его вызвал.
- [ ] Создан Space из трёх файлов, секрет `HF_TOKEN` на месте, по URL открывается чат.
- [ ] Ссылка на ваш Space записана.
- [ ] Токен нигде не захардкожен — только `.env` / Secrets.

Вывод одной фразой: что оказалось самым неочевидным в деплое.

_(Ваш вывод одной фразой здесь.)_